In [2]:
import pandas as pd
rating_train_sample = pd.read_csv('train_set.csv')
rating_test_sample = pd.read_csv('test_set.csv')
print("Train shape:", rating_train_sample.shape)
print("Test shape:", rating_test_sample.shape)

Train shape: (98597, 14)
Test shape: (172, 14)


In [1]:
import numpy as np
import sys
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
sys.path.append('../../')
from src.models.vectorDBbuild import VectorDBBuilder
from src.models.lightfm import LightFMTrainer
from src.models.hybrid_retrieval import RetrievalEngine
from src.models.newUser import NewUserRetriever

def run_full_pipeline(data, test_user_ids=None, debug=False):
    print("Training LightFM model")
    lightfm_model = LightFMTrainer()
    lightfm_model.fit_and_train(data, learning_rate=0.01, epochs=50)
    all_recommendations = lightfm_model.get_top_k(k=50)
    
    df_data = []
    for user_id, recs in all_recommendations.items():
        df_data.append({
            'user_id': user_id,
            'recommendations': recs
        })
    
    df = pd.DataFrame(df_data)
    
    # for user_id, recs in all_recommendations.items():
    #     for rec in recs:
    #         # Enhance with additional metadata if available
    #         enhanced_rec = {
    #             'user_id': user_id,
    #             'iid': rec['iid'],
    #             'title': rec.get('title', ''),
    #             'categories': rec.get('categories', []),
    #             'description': rec.get('description', ''),
    #             'cf_score': rec['score']
    #         }
    #         df_data.append(enhanced_rec)
    
    # df = pd.DataFrame(df_data)
    
    # build vector DB
    print("Building vector database...")
    vector_builder = VectorDBBuilder()
    embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
    df = vector_builder.preprocess_for_augmentation(df)
    items_df = vector_builder.extract_items_metadata(df)
    chroma_db = vector_builder.build_from_dataframe(items_df)
    
    # set up retrieval engine 
    retriever = RetrievalEngine(None, embedder)
    retriever.chroma_db = chroma_db
    # llm_reranker = LLMReranker()
    
    return {
        'df': df,
        'chroma_db': chroma_db,
        'retriever': retriever,
        # 'llm_reranker': llm_reranker,
        # 'top_k_results': top_k_results,
        'lightfm_model': lightfm_model
    }

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [3]:
def get_recommendations_for_user(pipeline_objects, user_id, query, debug=False):
    df = pipeline_objects['df']
    chroma_db = pipeline_objects['chroma_db']
    retriever = pipeline_objects['retriever']
    # llm_reranker = pipeline_objects['llm_reranker']
    #top_k_results = pipeline_objects['top_k_results']
    
    user_type = "existing" if check_user_type(user_id, df) else "new"
    
    if user_type == "existing":
        user_recs = df[df['user_id'] == user_id]['recommendations'].iloc[0]
        # user_recs = top_k_results[user_id]
        
        augmented = []
        for rec in user_recs:
            # get additional context from vector database
            try:
                chunks = chroma_db.similarity_search(
                    query=rec.get('combined_text', rec.get('title', '')),
                    k=3,
                    filter={"iid": rec['iid']}
                )
                augmented_text = f"{rec.get('combined_text', '')}\nRelated Context:\n{' '.join([chunk.page_content for chunk in chunks])}"
            except:
                # fallback if vector search fails
                augmented_text = rec.get('combined_text', '')
            
            augmented.append({
                **rec,
                "combined_text": rec.get('combined_text', ''),
                "augmented_text": augmented_text
            })
        
        # if debug:
        #     print("=== After Augmentation ===")
        #     print(f"Found {len(augmented)} items")
        #     for i, item in enumerate(augmented[:3], 1):
        #         print(f"{i}. {item['title']} - {item.get('combined_text', '')[:100]}...")
        
        # Hybrid retrieval
        scored = retriever.hybrid_retrieve(augmented, query)
        
        if debug:
            print("\n=== After Hybrid Retrieval ===")
            print(f"Top 10 items by hybrid score:")
            for i, item in enumerate(sorted(scored, key=lambda x: x.get('hybrid_score', 0), reverse=True)[:10], 1):
                print(f"{i}. {item['title']} (Hybrid: {item.get('hybrid_score', 0):.3f})")
                print(f"   Categories: {''.join(item.get('categories', []))}")
                description = item.get('description', '')
                if description and isinstance(description, str):
                    print(f"   Description: {description[:150]}...")
                # print(f"   Description: {item.get('description', '')[:150]}...\n")
        
        # MMR Diversification
        diversified = retriever.mmr_diversify(scored, query)
        
        if debug:
            print("\n=== After MMR Diversification ===")
            print(f"Top 10 diversified items:")
            for i, item in enumerate(diversified[:10], 1):
                print(f"{i}. {item['title']} (Hybrid: {item.get('hybrid_score', 0):.3f}, MMR: {item.get('mmr_score', 0):.3f})")
                print(f"   Categories: {''.join(item.get('categories', []))}")
                # print(f"   Description: {item.get('description', '')[:150]}...\n")
                description = item.get('description', '')
                if description and isinstance(description, str):
                    print(f"   Description: {description[:150]}...")
                print()
                
        # LLM Reranking
        # llm_result = llm_reranker.rerank(diversified, query, user_id, user_type)
        
        # if debug:
        #     print("\n=== Final LLM-Ranked Results ===")
        #     print(f"Top 10 final recommendations:")
        #     for i, item in enumerate(llm_result['parsed_items'][:10], 1):
        #         print(f"{i}. {item['title']} (ID: {item['iid']})")
        #         if 'hybrid_score' in item:
        #             print(f"   Hybrid Score: {item['hybrid_score']:.3f}")
        #         if 'mmr_score' in item:
        #             print(f"   MMR Score: {item['mmr_score']:.3f}")
        #         print("   Reason:")
        #         if 'llm_reason' in item:
        #             for reason_line in item['llm_reason'].split('\n'):
        #                 print(f"     {reason_line}")
        #         else:
        #             print("     - No reasoning provided")
        #         print() 
        
        return diversified[:10]
        # llm_result['parsed_items'][:10]

In [10]:
 # else:
    #     # New user handling
    #     new_user_retriever = NewUserRetriever(chroma_db, retriever.embedder)
    #     base_results = sorted(
    #         new_user_retriever.retrieve(query),
    #         key=lambda x: x.get('semantic_score', 0),
    #         reverse=True
    #     )
        
    #     llm_result = llm_reranker.rerank(base_results[:50], query, user_id, "new")
        
    #     if debug:
    #         print("\n=== Raw LLM Output ===")
    #         print(llm_result['llm_output'])
            
    #         print("\n=== Final Recommendations ===")
    #         for i, item in enumerate(llm_result['parsed_items'][:10], 1):
    #             print(f"{i}. {item['title']} (ID: {item['iid']})")
    #             print(f"   Categories: {', '.join(item.get('categories', []))}")
    #             print(f"   Semantic Score: {item.get('semantic_score', 0):.2f}")
    #             print("   Reason:")
    #             if 'llm_reason' in item:
    #                 for reason_line in item['llm_reason'].split('\n'):
    #                     print(f"     {reason_line}")
    #             else:
    #                 print("     • No reasoning provided")
    #             print()
        
    #     return llm_result['parsed_items'][:10]

In [4]:
def check_user_type(user_id, df):
    return user_id in df['user_id'].values

In [12]:
# old, eval for only 1 user
def main():
    # Load data - adjust paths as needed
    # try:
    #     train_data = pd.read_parquet("data/train/interactions.parquet")
    # except:
    #     # Fallback to CSV if parquet not available
    test_data = pd.read_csv("test_set.csv")
    train_data = pd.read_csv("train_set.csv")
    
    # run the full pipeline
    print("Running full recommendation pipeline...")
    pipeline_objects = run_full_pipeline(test_data)

    df = pipeline_objects['df']
    test_users = df['user_id'].unique().tolist() 
    print(f"Available users: {test_users[:5]}...") 
    
    # test with a specific user and query
    # test_users = list(pipeline_objects['top_k_results'].keys())
    
    if test_users:
        test_user_id = test_users[4]  # first user
        test_query = "relaxing jazz with piano"
        
        print(f"Testing pipeline for user {test_user_id} with query: {test_query}")
        recommendations = get_recommendations_for_user(
            pipeline_objects, test_user_id, test_query, debug=True
        )

        # Print final recommendations
        print("\n" + "="*50)
        print("FINAL RECOMMENDATIONS:")
        print("="*50)
        
        # print("Final recommendations:")
        for i, rec in enumerate(recommendations, 1):
            print(f"{i}. {rec.get('title', 'No title')} (ID: {rec.get('iid', 'No ID')})")
            print(f"   Hybrid Score: {rec.get('hybrid_score', 0):.3f}")
            print(f"   MMR Score: {rec.get('mmr_score', 0):.3f}")
            if rec.get('categories'):
                print(f"   Categories: {''.join(rec.get('categories', []))}")
            description = rec.get('description', '')
            if description and isinstance(description, str):
                print(f"   Description: {description[:100]}...")
            # if rec.get('description'):
            #     print(f"   Description: {rec.get('description', '')[:100]}...")
            print()

        print("\n" + "="*50)
        print("EVALUATION RESULTS:")
        print("="*50)
        
        # get ground truth
        true_items = test_data[test_data['uid'] == test_user_id]['iid'].unique().tolist()
        user_history = train_data[train_data['uid'] == test_user_id]['iid'].unique().tolist()
        
        if true_items:
            relevance_metrics = calculate_relevance_metrics(true_items, recommendations, debug = True)
            diversity_metrics = calculate_diversity_metrics(recommendations)
            novelty_metrics = calculate_novelty_metrics(recommendations, user_history)
            
            print(f"User: {test_user_id}")
            print(f"Query: {test_query}")
            print(f"True items in test set: {len(true_items)}")
            print(f"User history items: {len(user_history)}")
            
            print("\n--- Relevance Metrics ---")
            for metric, value in relevance_metrics.items():
                print(f"{metric}: {value:.4f}")
            
            # print("\n--- Diversity Metrics ---")
            # for metric, value in diversity_metrics.items():
            #     if isinstance(value, float):
            #         print(f"{metric}: {value:.4f}")
            #     else:
            #         print(f"{metric}: {value}")
            
            print("\n--- Novelty Metrics ---")
            for metric, value in novelty_metrics.items():
                print(f"{metric}: {value:.4f}")

            print("\n--- Diversity Metrics ---")
            for metric, value in diversity_metrics.items():
                if isinstance(value, float):
                    print(f"{metric}: {value:.4f}")
                else:
                    print(f"{metric}: {value}")
            
            
        else:
            print(f"No ground truth found for user {test_user_id} in test set")
            
    else:
        print("No test users found in recommendations")

if __name__ == "__main__":
    main()

Running full recommendation pipeline...
Training LightFM model


100%|██████████████████████████████████████████████████████████████████████████| 153/153 [00:00<00:00, 176.24it/s]


Building vector database...
Available users: ['AHHQO5P6AV5OC6LRBETHDKXLCHVQ', 'AGBZZQMYM6LPCPPJPG7ELBF2D6RA', 'AF34723SYWSLE3XCZNNZY2NYQRAQ', 'AHTOYYE4SUTY23LJJ73N7UI4OOSQ', 'AHZFWGWVWMWUYWERL72QHZSY7DBQ']...
Testing pipeline for user AHZFWGWVWMWUYWERL72QHZSY7DBQ with query: relaxing jazz with piano

=== After Hybrid Retrieval ===
Top 10 items by hybrid score:
1. Jazz Samba VME - Remastered (Hybrid: 0.141)
   Categories: CDs & Vinyl, International Music, South & Central America, Brazil, Bossa Nova
   Description: Product Description, This is the 1st true bossa nova album by jazz artists - the one that began the wave. Stan Getz is recognized as master of the for...
2. The Very Best Of KC & The Sunshine Band (Hybrid: 0.093)
   Categories: CDs & Vinyl, Dance & Electronic, Disco
   Description: 18 track best of collection includes 3 bonus floor fillers 'Get Down Tonight' (Original Long Version), 'That's the Way (I Like It)' (Original Long Ver...
3. Blues from Laurel Canyon (Hybrid: 0.079)


In [8]:
# need to fix
from src.evaluation.diversity_metrics import calculate_diversity_metrics, clean_categories

def main():
    # Load data
    train_data = pd.read_csv("train_set.csv")
    test_data = pd.read_csv("test_set.csv")
    
    print("Data loaded successfully!")
    print(f"Train data shape: {train_data.shape}")
    print(f"Test data shape: {test_data.shape}")
    print(f"Unique users in train: {train_data['uid'].nunique()}")
    print(f"Unique users in test: {test_data['uid'].nunique()}")
    
    # Run pipeline on TRAIN data
    print("Running full recommendation pipeline on TRAIN data...")
    pipeline_objects = run_full_pipeline(train_data)

    # Get TEST users from TEST data
    test_users = test_data['uid'].unique().tolist()
    print(f"Total test users available: {len(test_users)}")
    print(f"Sample test users: {test_users[:5]}")
    
    # Filter test users to only those that exist in train data
    train_users = set(train_data['uid'].unique())
    qualified_test_users = [user for user in test_users if user in train_users]
    print(f"Test users with training history: {len(qualified_test_users)}")
    
    if not qualified_test_users:
        print("No test users found with training history!")
        return
    
    # Store metrics for all users
    all_relevance_metrics = []
    all_diversity_metrics = []
    all_novelty_metrics = []
    
    # Test query
    test_query = "pop music with piano"
    
    print(f"\nEvaluating {len(qualified_test_users)} TEST users with query: '{test_query}'")
    print("="*60)
    
    for i, user_id in enumerate(qualified_test_users):
        print(f"Evaluating test user {i+1}/{len(qualified_test_users)}: {user_id}")
        
        try:
            # Get recommendations for TEST user using trained pipeline
            recommendations = get_recommendations_for_user(
                pipeline_objects, user_id, test_query, debug=False
            )
            
            if not recommendations:
                print(f"  No recommendations for test user {user_id}")
                continue
            
            # Get ground truth from TEST data
            true_items = test_data[test_data['uid'] == user_id]['iid'].unique().tolist()
            
            # Get user history from TRAIN data (what the model was trained on)
            user_history = train_data[train_data['uid'] == user_id]['iid'].unique().tolist()
            
            if not true_items:
                print(f"  No ground truth in test set for user {user_id}")
                continue
            
            # Calculate metrics for this TEST user
            relevance_metrics = calculate_relevance_metrics(
                true_items, recommendations, k=10, debug=False
            )
            diversity_metrics = calculate_diversity_metrics(recommendations)
            # novelty_metrics = calculate_novelty_metrics(recommendations, user_history)
            
            # Store results
            relevance_metrics['user_id'] = user_id
            diversity_metrics['user_id'] = user_id
            # novelty_metrics['user_id'] = user_id
            
            all_relevance_metrics.append(relevance_metrics)
            all_diversity_metrics.append(diversity_metrics)
            # all_novelty_metrics.append(novelty_metrics)
            
            print(f"NDCG@10={relevance_metrics['NDCG@10']:.3f}, "
                  f"Hit={relevance_metrics['HitRate@10']}")
            
        except Exception as e:
            print(f"  ✗ Error evaluating test user {user_id}: {e}")
    
    # Calculate aggregate metrics
    print("\n" + "="*60)
    print("AGGREGATE EVALUATION RESULTS (ON TEST SET)")
    print("="*60)
    
    if all_relevance_metrics:
        # Relevance metrics
        aggregate_relevance = calculate_aggregate_relevance_metrics(all_relevance_metrics)
        
        print("\n RELEVANCE METRICS (Test Set Performance):")
        print(f"Test users evaluated: {aggregate_relevance['Total_Users']}")
        print(f"MAP@10: {aggregate_relevance['MAP@10']:.4f}")
        print(f"Mean NDCG@10: {aggregate_relevance['Mean_NDCG@10']:.4f} ± {aggregate_relevance['Std_NDCG@10']:.4f}")
        print(f"Mean MRR@10: {aggregate_relevance['Mean_MRR@10']:.4f}")
        print(f"Hit Rate@10: {aggregate_relevance['HitRate@10']:.4f}")
        print(f"Average true items per test user: {aggregate_relevance['Avg_True_Items_Per_User']:.2f}")
        print(f"Total relevant items recommended: {aggregate_relevance['Total_Relevant_Items']}")
        
        # Diversity metrics
        if all_diversity_metrics:
            aggregate_diversity = calculate_aggregate_diversity_metrics(all_diversity_metrics)
            print("\n DIVERSITY METRICS:")
            print(f"Mean Category Coverage: {aggregate_diversity['Mean_CategoryCoverage']:.4f}")
            print(f"Mean Unique Categories: {aggregate_diversity['Mean_UniqueCategories']:.2f}")
            print(f"Mean Entropy: {aggregate_diversity['Mean_Entropy']:.4f}")
            print(f"Mean Gini Index: {aggregate_diversity['Mean_GiniIndex']:.4f}")
        
        # Novelty metrics
        # if all_novelty_metrics:
        #     aggregate_novelty = calculate_aggregate_novelty_metrics(all_novelty_metrics)
        #     print("\n🆕 NOVELTY METRICS:")
            # print(f"Mean Novelty@10: {aggregate_novelty['Mean_Novelty@10']:.4f}")
            # print(f"Mean Serendipity@10: {aggregate_novelty['Mean_Serendipity@10']:.4f}")
        
        # Save detailed results
        save_detailed_results(all_relevance_metrics, all_diversity_metrics, all_novelty_metrics)
        
        print(f"\n Evaluation completed!")
        print(f"   Successfully evaluated {aggregate_relevance['Total_Users']}/{len(qualified_test_users)} test users")
        
    else:
        print(" No successful evaluations on test set!")

def save_detailed_results(relevance_metrics, diversity_metrics, novelty_metrics):
    """Save detailed per-user results to CSV"""
    import pandas as pd
    
    # Create DataFrames
    rel_df = pd.DataFrame(relevance_metrics)
    div_df = pd.DataFrame(diversity_metrics).add_prefix('div_')
    # nov_df = pd.DataFrame(novelty_metrics).add_prefix('nov_')
    
    # Merge on user_id
    detailed_df = rel_df.merge(div_df, left_on='user_id', right_on='div_user_id')
    detailed_df = detailed_df.merge(nov_df, left_on='user_id', right_on='nov_user_id')
    
    # Clean up duplicate columns
    detailed_df = detailed_df.drop(['div_user_id', 'nov_user_id'], axis=1)
    
    # Save to CSV
    detailed_df.to_csv("test_set_evaluation_results.csv", index=False)
    print(f" Detailed test set results saved to: test_set_evaluation_results.csv")

if __name__ == "__main__":
    main()

Data loaded successfully!
Train data shape: (98597, 14)
Test data shape: (172, 14)
Unique users in train: 57297
Unique users in test: 153
Running full recommendation pipeline on TRAIN data...
Training LightFM model


100%|███████████████████████████████████████████████████████████████████████████████████████| 57297/57297 [2:36:42<00:00,  6.09it/s]


Building vector database...


/Users/vynguyen/Documents/VSCode/Thesis/notebooks/amazon/../../src/models/vectorDBbuild.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)


Total test users available: 153
Sample test users: ['AHHQO5P6AV5OC6LRBETHDKXLCHVQ', 'AGBZZQMYM6LPCPPJPG7ELBF2D6RA', 'AF34723SYWSLE3XCZNNZY2NYQRAQ', 'AHTOYYE4SUTY23LJJ73N7UI4OOSQ', 'AHZFWGWVWMWUYWERL72QHZSY7DBQ']
Test users with training history: 153

Evaluating 153 TEST users with query: 'pop music with piano'
Evaluating test user 1/153: AHHQO5P6AV5OC6LRBETHDKXLCHVQ
  ✗ Error evaluating test user AHHQO5P6AV5OC6LRBETHDKXLCHVQ: name 'calculate_user_relevance_metrics' is not defined
Evaluating test user 2/153: AGBZZQMYM6LPCPPJPG7ELBF2D6RA
  ✗ Error evaluating test user AGBZZQMYM6LPCPPJPG7ELBF2D6RA: name 'calculate_user_relevance_metrics' is not defined
Evaluating test user 3/153: AF34723SYWSLE3XCZNNZY2NYQRAQ
  ✗ Error evaluating test user AF34723SYWSLE3XCZNNZY2NYQRAQ: name 'calculate_user_relevance_metrics' is not defined
Evaluating test user 4/153: AHTOYYE4SUTY23LJJ73N7UI4OOSQ
  ✗ Error evaluating test user AHTOYYE4SUTY23LJJ73N7UI4OOSQ: name 'calculate_user_relevance_metrics' is not defi

In [5]:
def prepare_evaluation_data(train_data, test_data, user_id):
    true_items = test_data[test_data['uid'] == user_id]['iid'].unique().tolist()
    user_history = train_data[train_data['uid'] == user_id]['iid'].unique().tolist()
    
    return {
        'true_items': true_items,      # for relevance 
        'user_history': user_history   # for novelty 
    }

In [6]:
from sklearn.metrics import ndcg_score
import numpy as np
from collections import Counter
import pandas as pd

# def calculate_relevance_metrics(true_items, recommendations, debug = False):
#     predicted_items = [rec['iid'] for rec in recommendations]
#     k = len(recommendations)
#     true_set = set(true_items)
#     predicted_set = set(predicted_items)
#     overlap = true_set.intersection(predicted_set)

#     if debug:
#         print(f"Overlap between true and predicted sets: {overlap}")
#         print(f"True set size: {len(true_set)}")
#         print(f"Predicted set size: {len(predicted_set)}")
        
#     predicted_score_dict = {}
#     for rank, iid in enumerate(predicted_items):
#         predicted_score_dict[iid] = 1 - (rank / len(predicted_items))

#     true_set = set(true_items)
#     predicted_relevance = [1 if item in true_set else 0 for item in predicted_items]
#     ideal_relevance = [1] * min(len(true_set), k) + [0] * max(0, k-len(true_set))

#     if debug:
#         print(f"Predicted relevance vector: {predicted_relevance}")
#         print(f"Ideal relevance vector: {ideal_relevance}")

#     try: 
#         max_len = max(len(predicted_relevance), len(ideal_relevance))
#         predicted_padded = predicted_relevance + [0]* (max_len - len(predicted_relevance))
#         ideal_padded = ideal_relevance + [0]* (max_len - len(ideal_relevance))
        
#         ndcg = ndcg_score([ideal_padded], [predicted_padded], k=10)
#     except Exception as e:
#         if debug:
#             print(f"NDCG calculation error: {e}")
#         ndcg = 0.0

#     true_positive = len(overlap)
#     precision = true_positive / k if k > 0  else 0
#     recall = true_positive / len(true_set) if true_set else 0

#     return {
#         'NDCG@10': ndcg,
#         'True_Positives': true_positive,
#         'Precision@10': precision,
#         'Recall@10': recall
#     }

In [7]:
def calculate_novelty_metrics(recommendations, user_history):
    seen_items = set(user_history)
    return {
        'Novelty@10': len([rec for rec in recommendations if rec['iid'] not in seen_items])/10,
        'Serendipity': len([rec for rec in recommendations if rec['iid'] not in seen_items and rec['hybrid_score'] > 0.4])/10
    }

In [5]:
from sklearn.metrics import ndcg_score
import numpy as np

def calculate_relevance_metrics(true_items, recommendations, k=10, debug=False):
    predicted_items = [rec['iid'] for rec in recommendations[:k]]
    true_set = set(true_items)

    relevance_score = [1 if iid in true_set else 0 for iid in predicted_items]
    
    # hit rate
    hit = 1 if any(relevance_score) else 0

    # MRR@k
    mrr = 0.0
    for rank, rel in enumerate(relevance_scores, start=1):
        if rel == 1:
            mrr = 1.0 / rank
            break
            
    # Precision@k
    precision = sum(relevance_score)/k

    # AP@k
    ap = 0.0
    num_relevant = 0
    for rank, rel in enumerate(relevance_score, start = 1):
        if rel == 1:
            num_relevant += 1
            ap += num_relevant/rank
    ap = ap/min(len(true_set), k) if true_set else 0

    # NDCG@k (using binary relevance)
    ideal_relevance = sorted(relevance_score, reverse=True)
    if sum(relevance_true) == 0:
        ndcg = 0.0
    else:
        ndcg = ndcg_score([ideal_relevance], [relevance_score], k=k)
    
    if debug:
        print(f"True items: {true_items}")
        print(f"Predicted items: {predicted_items}")
        print(f"Relevance scores: {relevance_score}")
        print(f"Hit: {hit}, Precision@{k}: {precision:.4f}, Recall@{k}: {recall:.4f}")
        print(f"AP@{k}: {ap:.4f}, NDCG@{k}: {ndcg:.4f}, MRR@{k}: {mrr:.4f}")
        
    return {
        f'HitRate@{k}': hit,
        f'AP@{k}': ap,
        f'NDCG@{k}': ndcg,
        f'MRR@{k}': mrr,
        'num_relevant': sum(relevance_score),
        'num_true_items': len(true_set)
    }

In [7]:
def calculate_aggregate_relevance_metrics(user_metrics_list, k=10):
    """
    Calculate aggregate metrics across all users
    """
    if not user_metrics_list:
        return {}
    
    # Convert to DataFrame for easier aggregation
    import pandas as pd
    df = pd.DataFrame(user_metrics_list)
    
    aggregate_metrics = {
        f'MAP@{k}': df[f'AP@{k}'].mean(),
        f'Mean_NDCG@{k}': df[f'NDCG@{k}'].mean(),
        f'Mean_MRR@{k}': df[f'MRR@{k}'].mean(),
        f'HitRate@{k}': df[f'HitRate@{k}'].mean(),
        'Total_Users': len(df),
        'Total_Relevant_Items': df['num_relevant'].sum(),
        'Avg_True_Items_Per_User': df['num_true_items'].mean()
    }
    
    # # Add standard deviations
    # aggregate_metrics.update({
    #     f'Std_NDCG@{k}': df[f'NDCG@{k}'].std(),
    #     f'Std_Precision@{k}': df[f'Precision@{k}'].std(),
    #     f'Std_Recall@{k}': df[f'Recall@{k}'].std()
    # })
    
    return aggregate_metrics

In [9]:
def save_pipeline(pipeline_objects, filepath="saved_pipeline.pkl"):
    """Save trained pipeline objects for later use"""
    # Don't save ChromaDB as it's already persisted
    # Save other objects that are expensive to recompute
    save_objects = {
        'df': pipeline_objects['df'],
        'top_k_results': pipeline_objects['top_k_results'],
        
    }
    
    with open(filepath, 'wb') as f:
        pickle.dump(save_objects, f)
    print(f" Pipeline saved to {filepath}")

def load_pipeline(filepath="saved_pipeline.pkl", chroma_db_path="./chroma_db"):
    """Load saved pipeline objects"""
    try:
        # Load ChromaDB
        from langchain.vectorstores import Chroma
        from langchain.embeddings import HuggingFaceEmbeddings
        
        embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
        chroma_db = Chroma(persist_directory=chroma_db_path, embedding_function=embedder)
        
        # Load other objects
        with open(filepath, 'rb') as f:
            saved_objects = pickle.load(f)
        
        # Recreate retriever and other components
        from src.models.hybrid_retrieval import RetrievalEngine
        retriever = RetrievalEngine(chroma_db, embedder)
        
        pipeline_objects = {
            'chroma_db': chroma_db,
            'retriever': retriever,
            **saved_objects
        }
        
        print(f" Pipeline loaded from {filepath}")
        return pipeline_objects
        
    except Exception as e:
        print(f" Error loading pipeline: {e}")
        return None

def main(use_saved_pipeline=True):
    # Load data
    train_data = pd.read_csv("train_set.csv")
    test_data = pd.read_csv("test_set.csv")
    
    print("Data loaded successfully!")
    print(f"Train data shape: {train_data.shape}")
    print(f"Test data shape: {test_data.shape}")
    print(f"Unique users in train: {train_data['uid'].nunique()}")
    print(f"Unique users in test: {test_data['uid'].nunique()}")
    
    # Check if we should use saved pipeline or create new one
    if use_saved_pipeline and os.path.exists("saved_pipeline.pkl"):
        print("Loading saved pipeline...")
        pipeline_objects = load_pipeline()
        
        if pipeline_objects is None:
            print("Failed to load pipeline, training new one...")
            pipeline_objects = run_full_pipeline(train_data)
            save_pipeline(pipeline_objects)
    else:
        # Run pipeline on TRAIN data
        print("Running full recommendation pipeline on TRAIN data...")
        pipeline_objects = run_full_pipeline(train_data)
        save_pipeline(pipeline_objects)

    # Get TEST users from TEST data
    test_users = test_data['uid'].unique().tolist()
    print(f"Total test users available: {len(test_users)}")
    print(f"Sample test users: {test_users[:5]}")
    
    # Filter test users to only those that exist in train data
    train_users = set(train_data['uid'].unique())
    qualified_test_users = [user for user in test_users if user in train_users]
    print(f"Test users with training history: {len(qualified_test_users)}")
    
    if not qualified_test_users:
        print("No test users found with training history!")
        return
    
    # Store metrics for all users
    all_relevance_metrics = []
    all_diversity_metrics = []
    
    # Test query
    test_query = "pop music with piano"
    
    print(f"\nEvaluating {len(qualified_test_users)} TEST users with query: '{test_query}'")
    print("="*60)
    
    for i, user_id in enumerate(qualified_test_users):
        print(f"Evaluating test user {i+1}/{len(qualified_test_users)}: {user_id}")
        
        try:
            # Get recommendations for TEST user using trained pipeline
            recommendations = get_recommendations_for_user(
                pipeline_objects, user_id, test_query, debug=False
            )
            
            if not recommendations:
                print(f"  No recommendations for test user {user_id}")
                continue
            
            # Get ground truth from TEST data
            true_items = test_data[test_data['uid'] == user_id]['iid'].unique().tolist()
            
            # Get user history from TRAIN data (what the model was trained on)
            user_history = train_data[train_data['uid'] == user_id]['iid'].unique().tolist()
            
            if not true_items:
                print(f"  No ground truth in test set for user {user_id}")
                continue
            
            # FIXED: Use the correct function name
            relevance_metrics = calculate_user_relevance_metrics(  
                true_items, recommendations, k=10, debug=False
            )
            diversity_metrics = calculate_diversity_metrics(recommendations)
            
            # Store results
            relevance_metrics['user_id'] = user_id
            diversity_metrics['user_id'] = user_id
            
            all_relevance_metrics.append(relevance_metrics)
            all_diversity_metrics.append(diversity_metrics)
            
            print(f"  ✓ NDCG@10={relevance_metrics['NDCG@10']:.3f}, "
                  f"Hit={relevance_metrics['HitRate@10']}")
            
        except Exception as e:
            print(f"  ✗ Error evaluating test user {user_id}: {e}")
    
    # Calculate aggregate metrics
    print("\n" + "="*60)
    print("AGGREGATE EVALUATION RESULTS (ON TEST SET)")
    print("="*60)
    
    if all_relevance_metrics:
        # Relevance metrics
        aggregate_relevance = calculate_aggregate_relevance_metrics(all_relevance_metrics)
        
        print("\n RELEVANCE METRICS (Test Set Performance):")
        print(f"Test users evaluated: {aggregate_relevance['Total_Users']}")
        print(f"MAP@10: {aggregate_relevance['MAP@10']:.4f}")
        print(f"Mean NDCG@10: {aggregate_relevance['Mean_NDCG@10']:.4f} ± {aggregate_relevance['Std_NDCG@10']:.4f}")
        print(f"Mean MRR@10: {aggregate_relevance['Mean_MRR@10']:.4f}")
        print(f"Hit Rate@10: {aggregate_relevance['HitRate@10']:.4f}")
        print(f"Average true items per test user: {aggregate_relevance['Avg_True_Items_Per_User']:.2f}")
        print(f"Total relevant items recommended: {aggregate_relevance['Total_Relevant_Items']}")
        
        # Diversity metrics
        if all_diversity_metrics:
            aggregate_diversity = calculate_aggregate_diversity_metrics(all_diversity_metrics)
            print("\n DIVERSITY METRICS:")
            print(f"Mean Category Coverage: {aggregate_diversity['Mean_CategoryCoverage']:.4f}")
            print(f"Mean Unique Categories: {aggregate_diversity['Mean_UniqueCategories']:.2f}")
            print(f"Mean Entropy: {aggregate_diversity['Mean_Entropy']:.4f}")
            print(f"Mean Gini Index: {aggregate_diversity['Mean_GiniIndex']:.4f}")
        
        # Save detailed results
        save_detailed_results(all_relevance_metrics, all_diversity_metrics)
        
        print(f"\n Evaluation completed!")
        print(f"   Successfully evaluated {aggregate_relevance['Total_Users']}/{len(qualified_test_users)} test users")
        
    else:
        print(" No successful evaluations on test set!")

def save_detailed_results(relevance_metrics, diversity_metrics):
    rel_df = pd.DataFrame(relevance_metrics)
    div_df = pd.DataFrame(diversity_metrics).add_prefix('div_')
    
    # Merge on user_id
    detailed_df = rel_df.merge(div_df, left_on='user_id', right_on='div_user_id')
    
    # Clean up duplicate columns
    detailed_df = detailed_df.drop(['div_user_id'], axis=1)
    
    # Save to CSV
    detailed_df.to_csv("test_set_evaluation_results.csv", index=False)
    print(f" Detailed test set results saved to: test_set_evaluation_results.csv")


In [9]:
def clean_categories(categories_list):
    print(f"  clean_categories input: {categories_list} (type: {type(categories_list)})")
    
    if not categories_list:
        print("  → Empty categories list")
        return []
    
    if isinstance(categories_list, str):
        print(f"Input is string, splitting by comma")
        # split by comma if it's a string
        categories_list = [cat.strip() for cat in categories_list.split(',')]
    
    if not isinstance(categories_list, list):
        print(f"Not a list, returning empty")
        return []
    
    # Generic categories that don't provide meaningful diversity information
    generic_categories = {
        'CDs & Vinyl', 'Digital Music', 'Music', 'Albums', 
        'CDs', 'Vinyl'
    }
    
    cleaned = []
    for i, category in enumerate(categories_list):
        print(f"Processing category {i}: '{category}'")
        
        if isinstance(category, str):
            category = category.strip()
            if category not in generic_categories:
                cleaned.append(category)
            #     print(f"Added: '{category}'")
            # else:
            #     print(f"Skipped (generic): '{category}'")
        else:
            print(f"Skipped (not string): '{category}'")
    
    print(f"Final cleaned categories: {cleaned}")
    return cleaned

In [16]:
def calculate_diversity_metrics(recommendations):
    # print(f"\n=== DEBUGGING DIVERSITY METRICS ===")
    
    if not recommendations:
        print("No recommendations provided!")
        return {
            'CategoryCoverage': 0,
            'UniqueCategories': 0,
            'Entropy': 0,
            'GiniIndex': 0,
            'GenreDiversity': 0
        }
    
    # print raw categories from each recommendation
    # print("\n--- Raw categories from each recommendation ---")
    # for i, rec in enumerate(recommendations):
    #     raw_categories = rec.get('categories', [])
    #     print(f"Rec {i+1}: {raw_categories} (type: {type(raw_categories)})")
    
    all_categories = []
    
    for i, rec in enumerate(recommendations):
        raw_categories = rec.get('categories', [])
        # print(f"\nProcessing recommendation {i+1}:")
        # print(f"- Raw categories: {raw_categories}")
        
        cleaned_cats = clean_categories(raw_categories)
        # print(f"- Cleaned categories: {cleaned_cats}")
        
        all_categories.extend(cleaned_cats)
    
    # print(f"\nAll cleaned categories: {all_categories}")
    # print(f"Total category instances: {len(all_categories)}")
    
    if not all_categories:
        # print("WARNING: No categories found after cleaning!")
        return {
            'CategoryCoverage': 0,
            'UniqueCategories': 0,
            'Entropy': 0,
            'GiniIndex': 0,
            'GenreDiversity': 0
        }
    
    category_counts = Counter(all_categories)
    # print(f"Category counts: {dict(category_counts)}")
    
    # calculate metrics
    n_recommendations = len(recommendations)
    unique_categories = len(category_counts)
    
    category_coverage = unique_categories / n_recommendations
    entropy = 0
    gini = 0
    
    total_categories = len(all_categories)
    
    # entropy
    for category, count in category_counts.items():
        p = count / total_categories
        entropy -= p * np.log2(p) if p > 0 else 0
        # print(f"  Category '{category}': count={count}, probability={p:.3f}")
    
    # Gini index
    gini = 1 - sum((count / total_categories) ** 2 for count in category_counts.values())
    
    # print(f"\nFinal metrics:")
    # print(f"  UniqueCategories: {unique_categories}")
    
    # print(f"  CategoryCoverage: {category_coverage:.4f}")
    # print(f"  Entropy: {entropy:.4f}")
    # print(f"  GiniIndex: {gini:.4f}")
    
    return {
        'CategoryCoverage': category_coverage,
        'UniqueCategories': unique_categories,
        'Entropy': entropy,
        'GiniIndex': gini
    }

In [13]:
def calculate_aggregate_diversity_metrics(all_diversity_metrics):
    """
    Calculate aggregate diversity metrics across all users
    """
    if not all_diversity_metrics:
        return {}
    
    df = pd.DataFrame(all_diversity_metrics)
    
    return {
        'Mean_CategoryCoverage': df['CategoryCoverage'].mean(),
        'Mean_UniqueCategories': df['UniqueCategories'].mean(),
        'Mean_Entropy': df['Entropy'].mean(),
        'Mean_GiniIndex': df['GiniIndex'].mean(),
        'Std_CategoryCoverage': df['CategoryCoverage'].std(),
        'Std_UniqueCategories': df['UniqueCategories'].std()
    }

In [ ]:
def complete_recommendation_pipeline(user_id, query, debug=False):
    """
    End-to-end pipeline from LightFM candidates to final recommendations
    """
    # 1. Check if user exists in LightFM training data
    if user_id in fitted_data['user_ids']:
        # Existing user - get LightFM candidates then full pipeline
        user_index = fitted_data['user_ids'].index(user_id)
        
        # Get LightFM top-K candidates
        lightfm_scores = model.predict(user_ids=user_index, item_ids=list(range(fitted_data['n_items'])))
        top_indices = np.argsort(lightfm_scores)[::-1][:50]
        lightfm_candidates = [{
            'iid': fitted_data['item_ids'][idx],
            'title': fitted_data['item_features'].get('title', {}).get(fitted_data['item_ids'][idx], ''),
            'cf_score': float(lightfm_scores[idx]),
            'source': 'lightfm'
        } for idx in top_indices]
        
        # Convert to your pipeline format
        augmented_candidates = []
        for candidate in lightfm_candidates:
            # Get additional metadata from your items_df
            item_meta = items_df[items_df['iid'] == candidate['iid']].iloc[0]
            augmented_candidates.append({
                **candidate,
                'categories': item_meta.get('categories', []),
                'description': item_meta.get('description', ''),
                'combined_text': f"Title: {item_meta['title']}\nCategories: {item_meta.get('categories', '')}\nDescription: {item_meta.get('description', '')}"
            })
        
        # Run through your hybrid pipeline
        return get_recommendations(user_id, query, debug=debug)
    
    else:
        # New user - semantic search only
        retriever = NewUserRetriever(chroma_db, embedder)
        base_results = sorted(
            retriever.retrieve(query),
            key=lambda x: x['semantic_score'],
            reverse=True
        )
        
        llm_result = llm_rerank(base_results[:50], query, user_id, "new")
        return llm_result['parsed_items'][:10]